In [1]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [ ]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_5_try_0.pickle

In [3]:
%%RecordEvent
%%time
### cell 6 ###

def combine_subset_of_data_from_multiple_years_18(question_of_interest, include_2017=False):
    """
    Returns a DataFrame with columns ['percentage', 'year', question_of_interest]
    by computing the percent share of each value in question_of_interest
    for each year’s DataFrame.
    """
    # map each year label to its DataFrame
    year_dfs = [
        ('2022', responses_df_2022_cell10),
        ('2021', responses_df_2021),
        ('2020', responses_df_2020),
        ('2019', responses_df_2019_cell10),
        ('2018', responses_df_2018_cell10),
    ]
    if include_2017:
        year_dfs.append(('2017', responses_df_2017))

    records = []
    for year, df in year_dfs:
        s = (
            df[question_of_interest]
              .value_counts(dropna=False, normalize=True)
              .mul(100)
              .round(1)
              .sort_index()
        )
        temp = s.to_frame(name='percentage')
        temp['year'] = year
        temp[question_of_interest] = temp.index
        records.append(temp[['percentage', 'year', question_of_interest]])

    return pd.concat(records, ignore_index=True)


# --- Standardize raw DataFrames ---
question_name_alternate = 'Country'
question_name_alternate_cell18 = 'CurrentJobTitleSelect'

responses_df_2022_cell10.rename(
    columns={'United Kingdom (UK)': 'United Kingdom of Great Britain and Northern Ireland'},
    inplace=True
)
responses_df_2022_cell10.replace(
    'United Kingdom (UK)',
    'United Kingdom of Great Britain and Northern Ireland',
    inplace=True
)
responses_df_2017[question_name_alternate].replace(
    {
        'United States': 'United States of America',
        "People 's Republic of China": 'China',
        'United Kingdom': 'United Kingdom of Great Britain and Northern Ireland'
    },
    inplace=True
)
responses_df_2017.rename(
    columns={
        question_name_alternate: 'In which country do you currently reside?',
        question_name_alternate_cell18: (
            'Select the title most similar to your current role (or most recent title if retired): - Selected Choice'
        )
    },
    inplace=True
)

# limit to a subset of countries
subset_of_countries = [
    'Other', 'India', 'United States of America', 'Brazil', 'Nigeria',
    'Pakistan', 'Japan', 'China', 'Egypt', 'Indonesia', 'Mexico',
    'Turkey', 'Russia'
]
question_name = 'In which country do you currently reside?'
for df in [
    responses_df_2017,
    responses_df_2018_cell10,
    responses_df_2019_cell10,
    responses_df_2020,
    responses_df_2021,
    responses_df_2022_cell10
]:
    df.loc[~df[question_name].isin(subset_of_countries), question_name] = 'Other'

# preserve this variable for backwards compatibility/tests
title_for_x_axis = ''

# --- Combine and post-process ---
country_df_combined = combine_subset_of_data_from_multiple_years_18(
    question_name,
    include_2017=True
)

country_df_combined_cell18 = country_df_combined.sort_values(
    by=['year', 'percentage'],
    ascending=True
)
# final UK label correction
country_df_combined_cell18[question_name] = (
    country_df_combined_cell18[question_name]
      .replace('United Kingdom of Great Britain and Northern Ireland', 'United Kingdom')
)

country_df_combined_cell18.info()

/opt/conda/envs/elastic/lib/python3.10/site-packages/IPython/core/interactiveshell.py:2362: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  result = fn(*args, **kwargs)


<class 'pandas.core.frame.DataFrame'>
Index: 78 entries, 67 to 3
Data columns (total 3 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   percentage                                 78 non-null     float64
 1   year                                       78 non-null     object 
 2   In which country do you currently reside?  78 non-null     object 
dtypes: float64(1), object(2)
memory usage: 2.4+ KB
CPU times: user 50.1 ms, sys: 0 ns, total: 50.1 ms
Wall time: 49.9 ms


In [ ]:
%Checkpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_6_try_2.pickle